# Capstone 3: Write Like Shakespeare (Solution)

*Module 4 capstone. ML & NLP by Data Trainers LLC.*

This is the complete solution notebook. It implements **both paths** end-to-end:

- **Path A**: Character-level GRU trained from scratch on Tiny Shakespeare.
- **Path B**: GPT-2 fine-tuned on Tiny Shakespeare with HuggingFace Trainer.

At the end, a side-by-side comparison table summarizes trade-offs.

## What this solution covers

1. Complete char vocabulary and CharRNN implementation.
2. Full training loop with gradient clipping, loss tracking, validation BPC.
3. Temperature-controlled generation with `torch.multinomial`.
4. GPT-2 tokenization, dataset construction, Trainer fine-tuning.
5. Generation with `model.generate()` (top-p sampling).
6. Base vs fine-tuned comparison.
7. Technical memo and non-technical pitch templates.
8. Side-by-side comparison table with trade-off analysis.

## Scenario Recap

A literature startup needs a Shakespeare remixer for their creative writing app. We build, train, evaluate, and demo a Shakespeare text generator on the ~1.1 MB Tiny Shakespeare corpus, implementing both paths and evaluating quantitatively so the engineering team can compare them head-to-head.

## Section 0: Environment Setup

In [ ]:
# Install required packages (run this first in Google Colab)
!pip install -q torch transformers datasets accelerate evaluate matplotlib numpy

In [ ]:
# Core imports -- grouped by purpose (visualization -> data -> model -> training)
import os
import math
import time
import random
import urllib.request
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Reproducibility -- set all seeds so results are deterministic across runs.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Device -- use GPU if available, fall back to CPU.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"PyTorch version: {torch.__version__}")
print("\nEnvironment setup complete.")

## Section 1: Load Tiny Shakespeare

In [ ]:
# Load Tiny Shakespeare
# ~1.1 MB plain text from Karpathy's char-rnn repository (public domain)
SHAKESPEARE_URL = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'

if not os.path.exists('shakespeare.txt'):
    print("Downloading Tiny Shakespeare...")
    urllib.request.urlretrieve(SHAKESPEARE_URL, 'shakespeare.txt')
    print("Done.")

with open('shakespeare.txt', 'r', encoding='utf-8') as f:
    full_text = f.read()

print(f"Total characters: {len(full_text):,}")
print(f"Unique characters: {len(set(full_text))}")
print(f"Newlines: {full_text.count(chr(10)):,}")
print(f"\nFirst 400 characters:")
print(full_text[:400])

## Path A: Character-Level GRU

### Path A Hyperparameters

In [ ]:
# Path A hyperparameters
CHAR_SEQ_LEN    = 100    # sliding window length
CHAR_EMB_DIM    = 64     # embedding dimension (small vocab -> small embedding)
CHAR_HIDDEN     = 256    # GRU hidden size
CHAR_NUM_LAYERS = 2      # stacked GRU layers
CHAR_DROPOUT    = 0.2    # dropout applied between GRU layers
CHAR_BATCH_SIZE = 64
CHAR_EPOCHS     = 10
CHAR_LR         = 1e-3
CHAR_CLIP       = 1.0    # gradient clipping -- prevents exploding gradients in RNNs
GEN_MAX_NEW     = 200    # characters to generate per call
GEN_TEMP        = 0.8    # default temperature
SEED_PHRASE     = 'ROMEO:'

print("Path A hyperparameters set.")

### Solution A1: Character Vocabulary

In [ ]:
# Solution A1: build char vocab and encode the full text as integer IDs.

# 1. Sorted unique characters -- sorted() ensures deterministic order.
chars = sorted(set(full_text))   # list of ~65 unique chars

# 2. String-to-int and int-to-string mappings.
stoi = {c: i for i, c in enumerate(chars)}   # e.g. '\n' -> 0, ' ' -> 1, '!' -> 2, ...
itos = {i: c for c, i in stoi.items()}       # reverse lookup

vocab_size = len(chars)

# 3. Encode the entire text -- each char becomes its integer index.
all_ids = torch.tensor([stoi[c] for c in full_text], dtype=torch.long)

print(f"Vocabulary size: {vocab_size}")
print(f"Encoded text length: {len(all_ids):,} token IDs")
print(f"Sample chars: {chars[:10]}")
print(f"First 10 IDs: {all_ids[:10].tolist()}")

# ln(65) ~ 4.17 is the expected initial loss at random init, because all 65
# classes are equally likely at first. Handy sanity check for epoch 1.
print(f"\nExpected initial loss (random init): {math.log(vocab_size):.3f}   (= ln({vocab_size}))")

### Solution A2: CharDataset

In [ ]:
# Solution A2: Sliding-window dataset for character-level LM

class CharDataset(Dataset):
    """Yields (input_window, target_window) pairs from a char ID tensor.

    For position i:
      x = ids[i : i+seq_len]        <- input characters
      y = ids[i+1 : i+seq_len+1]    <- target = x shifted right by 1

    At every position t in the window, y[t] is the character the model
    should predict after seeing x[:t+1].
    """
    def __init__(self, ids, seq_len):
        self.ids     = ids
        self.seq_len = seq_len

    def __len__(self):
        # Total valid starting positions: len - seq_len.
        # (last window ends at index len-1 which is the target for len-2)
        return len(self.ids) - self.seq_len

    def __getitem__(self, i):
        x = self.ids[i     : i + self.seq_len]       # (seq_len,)
        y = self.ids[i + 1 : i + self.seq_len + 1]   # (seq_len,) -- shifted right by 1
        return x, y

# 90/10 train/val split at the character level
split_idx = int(0.9 * len(all_ids))
train_ids = all_ids[:split_idx]
val_ids   = all_ids[split_idx:]

train_ds_a = CharDataset(train_ids, CHAR_SEQ_LEN)
val_ds_a   = CharDataset(val_ids,   CHAR_SEQ_LEN)

train_loader_a = DataLoader(train_ds_a, batch_size=CHAR_BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader_a   = DataLoader(val_ds_a,   batch_size=CHAR_BATCH_SIZE, shuffle=False, num_workers=0)

# Sanity check: verify x and y are shifted by exactly 1 character
x_b, y_b = next(iter(train_loader_a))
print(f"Batch x shape: {x_b.shape}")
print(f"Batch y shape: {y_b.shape}")
print(f"x[:20]: {''.join(itos[i] for i in x_b[0, :20].tolist())}")
print(f"y[:20]: {''.join(itos[i] for i in y_b[0, :20].tolist())}")
# x and y should differ by exactly 1 char (y is the next-char for each position in x)

### Solution A3: CharRNN Model

In [ ]:
# Solution A3: CharRNN -- Embedding -> GRU -> Linear

class CharRNN(nn.Module):
    """
    Character-level language model using a stacked GRU.

    Architecture:
        Embedding(vocab_size, emb_dim)           -> (B, T, emb_dim)
        GRU(emb_dim, hidden, num_layers, ...)    -> (B, T, hidden)
        Linear(hidden, vocab_size)               -> (B, T, vocab_size)  [logits]

    The hidden state h is threaded through forward() so generation can
    run one token at a time while maintaining context.
    """
    def __init__(self, vocab_size, emb=CHAR_EMB_DIM, hidden=CHAR_HIDDEN,
                 layers=CHAR_NUM_LAYERS, dropout=CHAR_DROPOUT):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb)

        # batch_first=True so inputs are (B, T, features).
        # dropout only applies BETWEEN layers, not after the last one.
        self.gru = nn.GRU(
            input_size  = emb,
            hidden_size = hidden,
            num_layers  = layers,
            dropout     = dropout if layers > 1 else 0.0,
            batch_first = True,
        )

        self.fc = nn.Linear(hidden, vocab_size)

    def forward(self, x, h=None):
        """
        x : (B, T) integer token IDs
        h : (num_layers, B, hidden) or None -- previous hidden state

        Returns:
          logits : (B, T, vocab_size) -- raw scores (NOT softmax)
          h_new  : (num_layers, B, hidden)
        """
        e          = self.emb(x)       # (B, T, emb_dim)
        out, h_new = self.gru(e, h)    # out: (B, T, hidden); h_new: (layers, B, hidden)
        logits     = self.fc(out)      # (B, T, vocab_size)
        return logits, h_new

# Instantiate and move to device
model_a = CharRNN(vocab_size).to(device)
print(model_a)
n_params = sum(p.numel() for p in model_a.parameters() if p.requires_grad)
print(f"\nTrainable parameters: {n_params:,}")

# Quick forward pass to verify shapes
x_test         = torch.randint(0, vocab_size, (4, CHAR_SEQ_LEN)).to(device)
logits_test, h = model_a(x_test)
print(f"Output logits shape: {logits_test.shape}   (expect: [4, {CHAR_SEQ_LEN}, {vocab_size}])")
print(f"Hidden state shape:  {h.shape}        (expect: [{CHAR_NUM_LAYERS}, 4, {CHAR_HIDDEN}])")

### Solution A4: Training Loop

In [ ]:
# Solution A4: Full training loop with gradient clipping

criterion_a = nn.CrossEntropyLoss()   # expects (N, C) logits and (N,) targets
optimizer_a = torch.optim.Adam(model_a.parameters(), lr=CHAR_LR)

history_a = {'train_loss': [], 'val_loss': [], 'val_ppl': []}
t0 = time.time()

for epoch in range(1, CHAR_EPOCHS + 1):
    # Training phase
    model_a.train()
    total_loss, n_batches = 0.0, 0

    for x_batch, y_batch in train_loader_a:
        x_batch = x_batch.to(device)   # (B, T)
        y_batch = y_batch.to(device)   # (B, T)

        optimizer_a.zero_grad()

        # Forward: logits shape (B, T, V)
        logits, _ = model_a(x_batch)

        # CrossEntropyLoss expects (N, C) and (N,) -- reshape to (B*T, V), (B*T,).
        loss = criterion_a(
            logits.view(-1, vocab_size),
            y_batch.view(-1)
        )

        loss.backward()

        # Gradient clipping: prevents the exploding gradients common in RNNs.
        # clip_grad_norm_ scales all gradients so their L2 norm <= CHAR_CLIP.
        torch.nn.utils.clip_grad_norm_(model_a.parameters(), CHAR_CLIP)

        optimizer_a.step()
        total_loss += loss.item()
        n_batches  += 1

    train_loss = total_loss / n_batches

    # Validation phase
    model_a.eval()
    val_loss_total, val_batches = 0.0, 0

    with torch.no_grad():
        for x_v, y_v in val_loader_a:
            x_v, y_v   = x_v.to(device), y_v.to(device)
            logits_v, _ = model_a(x_v)
            v_loss      = criterion_a(logits_v.view(-1, vocab_size), y_v.view(-1))
            val_loss_total += v_loss.item()
            val_batches    += 1

    val_loss = val_loss_total / val_batches
    val_ppl  = math.exp(val_loss)   # perplexity = e^loss

    history_a['train_loss'].append(train_loss)
    history_a['val_loss'].append(val_loss)
    history_a['val_ppl'].append(val_ppl)

    print(f"Epoch {epoch:02d} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | val_ppl={val_ppl:.2f}")

train_time_a = time.time() - t0
print(f"\nPath A training complete in {train_time_a/60:.1f} minutes.")

In [ ]:
# Plot training curves for Path A
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

axes[0].plot(history_a['train_loss'], marker='o', label='train')
axes[0].plot(history_a['val_loss'],   marker='s', label='val')
axes[0].set_title('Loss curves - Path A (Char GRU)')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Cross-entropy loss')
axes[0].legend()

axes[1].plot(history_a['val_ppl'], marker='o', color='#FF7F50')
axes[1].set_title('Validation perplexity - Path A')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Perplexity')

plt.tight_layout()
plt.show()

### Solution A5: Text Generation with Temperature

In [ ]:
# Solution A5: Temperature sampling for autoregressive generation

@torch.no_grad()
def generate_char(model, seed, max_new=GEN_MAX_NEW, temp=GEN_TEMP):
    """
    Autoregressively generate `max_new` characters after `seed`.

    Temperature parameter controls the sharpness of the distribution:
      temp < 1.0 -> sharper (more conservative, lower entropy)
      temp > 1.0 -> flatter (more creative, higher entropy, more gibberish)
    """
    model.eval()

    # 1. Encode the seed text into integer IDs.
    seed_ids     = [stoi[c] for c in seed]
    input_tensor = torch.tensor(seed_ids, dtype=torch.long).unsqueeze(0).to(device)  # (1, L)

    # 2. Run the seed through the model to prime the hidden state.
    #    The last logit is the first prediction.
    logits, h = model(input_tensor)     # logits: (1, L, V); h: (layers, 1, hidden)

    generated  = list(seed)
    last_logit = logits[0, -1, :]       # (vocab_size,) -- logit for the char after seed

    for _ in range(max_new):
        # 3a. Temperature scaling: divide logits by temperature.
        #    High temp -> logits closer together -> flatter softmax -> more random.
        scaled = last_logit / temp

        # 3b. Convert to probability distribution.
        probs = F.softmax(scaled, dim=-1)   # (vocab_size,)

        # 3c. Sample one token ID according to the distribution.
        next_id = torch.multinomial(probs, num_samples=1)   # shape: (1,)

        # 3d. Decode the sampled ID back to a character.
        generated.append(itos[next_id.item()])

        # 3e. Feed the new token back as the next input (one step at a time).
        x_next         = next_id.unsqueeze(0)   # (1, 1) -- batch=1, time=1
        logits, h      = model(x_next, h)       # thread hidden state for context
        last_logit     = logits[0, -1, :]

    return ''.join(generated)

# Generate at default temperature
gen_a = generate_char(model_a, SEED_PHRASE, max_new=GEN_MAX_NEW, temp=GEN_TEMP)
print("=" * 60)
print(f"Seed: '{SEED_PHRASE}' | Temperature: {GEN_TEMP}")
print("=" * 60)
print(gen_a)
print("=" * 60)

In [ ]:
# Temperature sweep -- shows how temp controls creativity vs coherence.
# There's a sweet spot around 0.7-0.8 for Shakespeare.

for temp in [0.3, 0.7, 1.0, 1.5]:
    sample = generate_char(model_a, 'ROMEO:', max_new=80, temp=temp)
    print(f"\n--- Temperature = {temp} ---")
    print(sample)

# Expected observations:
# temp=0.3 -> repetitive but grammatically consistent (e.g., repeated phrases)
# temp=0.7 -> good balance, Shakespeare-like dialogue
# temp=1.0 -> more varied, occasionally strange word choices
# temp=1.5 -> creative but often incoherent, random-looking character sequences

### Solution A6: Bits-Per-Character

In [ ]:
# Solution A6: bits-per-character (BPC), the char-level metric for language model quality.
# BPC = avg_cross_entropy_loss / ln(2).
# Intuition: average number of bits needed to encode one character given the model.
# Lower is better. Shakespeare LSTM typically reaches 1.3-1.5 BPC with 10 epochs.
# State-of-the-art char-level models achieve ~1.0 BPC.

def compute_bpc(model, loader):
    """Compute bits-per-character on a held-out dataset."""
    model.eval()
    total_loss, n_batches = 0.0, 0

    with torch.no_grad():
        for x_b, y_b in loader:
            x_b, y_b    = x_b.to(device), y_b.to(device)
            logits, _   = model(x_b)
            loss        = criterion_a(
                logits.view(-1, vocab_size),
                y_b.view(-1)
            )
            total_loss += loss.item()
            n_batches  += 1

    avg_loss = total_loss / n_batches
    bpc = avg_loss / math.log(2)   # convert nats to bits
    return bpc, avg_loss

bpc_val, val_loss_a = compute_bpc(model_a, val_loader_a)
print(f"Validation cross-entropy: {val_loss_a:.4f} nats")
print(f"Validation BPC:           {bpc_val:.4f} bits/char")
print(f"Validation perplexity:    {math.exp(val_loss_a):.2f}")
print(f"\n(LSTM on Shakespeare typically achieves ~1.3-1.5 BPC after 10 epochs)")

# Save the model for the comparison section
torch.save(model_a.state_dict(), 'shakespeare_char_rnn.pt')
model_size_a = os.path.getsize('shakespeare_char_rnn.pt') / 1024 / 1024
print(f"\nModel saved (size: {model_size_a:.1f} MB)")

## Path B: Fine-tune GPT-2

### Path B Hyperparameters

In [ ]:
# Path B hyperparameters
MODEL_NAME    = 'gpt2'     # smallest GPT-2: 117M params, 12 layers, 768-d hidden
MAX_LENGTH    = 128        # token chunk size for fine-tuning
NUM_EPOCHS_B  = 3          # 3 epochs is plenty for style transfer on 1.1 MB
BATCH_SIZE_B  = 4          # small batch for Colab T4 memory
GRAD_ACCUM    = 4          # effective batch = 4 * 4 = 16, matches a standard run
LR_B          = 5e-5       # conservative LR for fine-tuning (don't destroy pretrained weights)
GEN_TEMP_B    = 0.8
GEN_TOP_P     = 0.9        # top-p (nucleus) sampling -- keeps only the top 90% probability mass
GEN_MAX_NEW_B = 200

print("Path B hyperparameters set.")

In [ ]:
# Import HuggingFace libraries for Path B
from transformers import (
    GPT2LMHeadModel, GPT2TokenizerFast,
    TrainingArguments, Trainer, DataCollatorForLanguageModeling
)
from datasets import Dataset as HFDataset

# Load GPT-2 tokenizer
# GPT-2 uses BPE (Byte Pair Encoding) with a vocabulary of ~50,257 subwords
tokenizer_b = GPT2TokenizerFast.from_pretrained(MODEL_NAME)

# GPT-2 has no padding token by default.
# Setting eos_token as pad_token is the standard workaround.
tokenizer_b.pad_token = tokenizer_b.eos_token

print(f"Tokenizer vocab size: {tokenizer_b.vocab_size:,}")
print(f"EOS/PAD token: '{tokenizer_b.eos_token}' (id={tokenizer_b.eos_token_id})")

# Demonstrate BPE tokenization on a Shakespeare line
sample = "To be, or not to be, that is the question."
tok    = tokenizer_b(sample)
print(f"\nSample: '{sample}'")
print(f"Token IDs: {tok['input_ids']}")
# Notice: BPE splits 'question' differently from how a char-level model would
print(f"Tokens: {[tokenizer_b.decode([t]) for t in tok['input_ids']]}")

### Solution B1: Tokenize Shakespeare and Build Training Chunks

In [ ]:
# Solution B1: Tokenize and chunk Shakespeare for causal LM training

# 1. Tokenize the entire text
# return_tensors='pt' gives us a PyTorch LongTensor
all_token_ids = tokenizer_b(full_text, return_tensors='pt')['input_ids'][0]
print(f"Total subword tokens: {len(all_token_ids):,}")
print(f"Compression ratio (chars/tokens): {len(full_text)/len(all_token_ids):.2f}x")
# BPE is ~4x more compressed than char-level: fewer steps per sequence

# 2. Split into non-overlapping chunks of MAX_LENGTH tokens
def chunk_tokens(token_ids, chunk_size):
    """Slice token_ids into non-overlapping chunks of chunk_size.
    The last partial chunk (< chunk_size) is discarded to keep batches uniform."""
    chunks = []
    for i in range(0, len(token_ids) - chunk_size, chunk_size):
        chunk = token_ids[i : i + chunk_size].tolist()
        chunks.append({'input_ids': chunk})
    return chunks

all_chunks = chunk_tokens(all_token_ids, MAX_LENGTH)
print(f"Total training chunks: {len(all_chunks):,}")

# 3. HuggingFace Dataset + 90/10 split
split_n     = int(0.9 * len(all_chunks))
train_hf    = HFDataset.from_list(all_chunks[:split_n])
val_hf      = HFDataset.from_list(all_chunks[split_n:])

print(f"Train chunks: {len(train_hf):,}   Val chunks: {len(val_hf):,}")

# Verify a chunk round-trips correctly
sample_chunk = train_hf[0]['input_ids']
print(f"\nFirst chunk decoded (first 100 chars):")
print(tokenizer_b.decode(sample_chunk[:30]))

### Solution B2: Fine-tune with HuggingFace Trainer

In [ ]:
# Solution B2: GPT-2 fine-tuning with HuggingFace Trainer

# 1. Load the pretrained GPT-2 model (117M params)
model_b = GPT2LMHeadModel.from_pretrained(MODEL_NAME)
model_b.resize_token_embeddings(len(tokenizer_b))  # re-size if pad token changes vocab size

total_params = sum(p.numel() for p in model_b.parameters())
print(f"GPT-2 parameters: {total_params:,}")

# 2. TrainingArguments controls the full training loop.
training_args = TrainingArguments(
    output_dir                  = './shakespeare_gpt2',
    num_train_epochs            = NUM_EPOCHS_B,
    per_device_train_batch_size = BATCH_SIZE_B,
    per_device_eval_batch_size  = BATCH_SIZE_B,
    gradient_accumulation_steps = GRAD_ACCUM,   # effective BS = 4 * 4 = 16
    learning_rate               = LR_B,
    eval_strategy               = 'epoch',      # evaluate once per epoch
    save_strategy               = 'no',         # skip saving checkpoints to save disk
    logging_steps               = 50,
    fp16                        = torch.cuda.is_available(),  # mixed precision on GPU
    seed                        = SEED,
    report_to                   = 'none',       # no wandb/tensorboard
)

# 3. DataCollator for Causal LM.
# mlm=False = CAUSAL LM (left-to-right, predict next token), which is GPT-2's objective.
# mlm=True would be MASKED LM (BERT's objective), not what we want here.
data_collator_b = DataCollatorForLanguageModeling(
    tokenizer = tokenizer_b,
    mlm       = False,
)

# 4. Trainer orchestrates the fine-tuning loop.
trainer_b = Trainer(
    model         = model_b,
    args          = training_args,
    train_dataset = train_hf,
    eval_dataset  = val_hf,
    data_collator = data_collator_b,
    tokenizer     = tokenizer_b,
)

print("Starting fine-tuning...")
t0_b = time.time()
trainer_b.train()
train_time_b = time.time() - t0_b
print(f"\nFine-tuning complete in {train_time_b/60:.1f} minutes.")

### Solution B3: Generate from Fine-tuned GPT-2

In [ ]:
# Solution B3: Text generation with top-p (nucleus) sampling
# Top-p sampling retains only the smallest set of tokens whose cumulative probability >= top_p.
# This avoids sampling from the very long tail of unlikely tokens.

def generate_gpt2(model, tokenizer, seed, max_new=GEN_MAX_NEW_B, temp=GEN_TEMP_B, top_p=GEN_TOP_P):
    """Generate text from GPT-2 using top-p nucleus sampling."""
    model.eval()
    gen_device = next(model.parameters()).device

    # Encode the seed prompt to token IDs
    inputs = tokenizer(seed, return_tensors='pt').to(gen_device)

    with torch.no_grad():
        output_ids = model.generate(
            inputs['input_ids'],
            max_new_tokens  = max_new,
            temperature     = temp,
            top_p           = top_p,
            do_sample       = True,     # must be True for temperature/top-p to apply
            pad_token_id    = tokenizer.eos_token_id,  # avoid warning about padding
        )

    # Decode: skip_special_tokens removes EOS/BOS tokens from the output
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

# Generate from fine-tuned model
model_b.to(device)
gen_b_finetuned = generate_gpt2(model_b, tokenizer_b, 'ROMEO:', max_new=GEN_MAX_NEW_B)
print("=" * 60)
print("Fine-tuned GPT-2 output:")
print("=" * 60)
print(gen_b_finetuned)

### Solution B4: Base vs Fine-tuned Comparison

In [ ]:
# Solution B4: Compare fine-tuned GPT-2 to base (un-fine-tuned) GPT-2
# Key question: does Shakespeare fine-tuning visibly shift the output style?

# Load a fresh copy of base GPT-2 (no Shakespeare weights)
base_gpt2 = GPT2LMHeadModel.from_pretrained(MODEL_NAME).to(device)

gen_b_base = generate_gpt2(base_gpt2, tokenizer_b, 'ROMEO:', max_new=GEN_MAX_NEW_B)

print("=" * 60)
print("BASE GPT-2 (no Shakespeare fine-tuning):")
print("=" * 60)
print(gen_b_base)
print()
print("=" * 60)
print("FINE-TUNED GPT-2 (after 3 epochs on Shakespeare):")
print("=" * 60)
print(gen_b_finetuned)

# GPT-2 likely saw some Shakespeare in its Common Crawl pretraining (Shakespeare is
# public domain and widely copied online), so the base model already knows some
# Elizabethan patterns. Fine-tuning makes them more consistent and less likely to
# drift into modern English mid-sentence.

### Solution B5: Compute Perplexity

In [ ]:
# Solution B5: Evaluate perplexity on the validation set

eval_results = trainer_b.evaluate()
eval_loss_b  = eval_results['eval_loss']
perplexity_b = math.exp(eval_loss_b)

print(f"Eval loss:    {eval_loss_b:.4f}")
print(f"Perplexity:   {perplexity_b:.2f}")

# GPT-2 perplexity is measured over SUBWORD tokens, not characters.
# You can't directly compare this number to the char-level BPC from Path A
# without re-normalizing. This is a common source of confusion.

# Save fine-tuned model
trainer_b.save_model('./shakespeare_gpt2_final')
model_size_b_mb = sum(
    os.path.getsize(os.path.join('./shakespeare_gpt2_final', f)) / 1024 / 1024
    for f in os.listdir('./shakespeare_gpt2_final')
    if f.endswith('.safetensors') or f.endswith('.bin')
) if os.path.exists('./shakespeare_gpt2_final') else float('nan')
print(f"\nFine-tuned model saved (~{model_size_b_mb:.0f} MB)")

## Section 4: Present Your Results

### Solution P1: Technical Memo (Example)

In [ ]:
# Solution P1: technical memo using actual numbers from both paths.

final_val_loss_a = history_a['val_loss'][-1]
final_val_ppl_a  = history_a['val_ppl'][-1]
bpc_val_a, _     = compute_bpc(model_a, val_loader_a)

memo = f"""
---
TECHNICAL MEMO: Shakespeare Generator -- Path Comparison
---

Path A: Character-Level GRU
  Architecture : Embedding(65, 64) -> GRU(64, 256, 2 layers) -> Linear(256, 65)
  Parameters   : {sum(p.numel() for p in model_a.parameters()):,}
  Training time: ~{train_time_a/60:.1f} min (10 epochs, T4 GPU)
  Val loss     : {final_val_loss_a:.4f} nats
  Val PPL      : {final_val_ppl_a:.2f}
  Val BPC      : {bpc_val_a:.4f} bits/char
  Model size   : {model_size_a:.1f} MB

Path B: Fine-tuned GPT-2
  Architecture : GPT-2 (117M params, 12-layer transformer)
  Fine-tuning  : 3 epochs, LR=5e-5, effective BS=16, fp16
  Training time: ~{train_time_b/60:.1f} min (3 epochs, T4 GPU)
  Val loss     : {eval_loss_b:.4f} nats (subword tokens)
  Val PPL      : {perplexity_b:.2f} (subword-level, not comparable to Path A)

Trade-off Summary
  Parameter count: Path A (~1M) vs Path B (~117M), 100x difference.
  Output quality : Path B is more coherent sentence-to-sentence.
  Controllability: Path A gives full architecture control; Path B is a black box.
  Data efficiency: Path B needs far less data (pretrained weights carry knowledge).
  Portability    : Path A model is tiny (~{model_size_a:.0f} MB); GPT-2 is ~{model_size_b_mb:.0f} MB.

Recommendation for production:
  Path B for quality demos; Path A for edge/embedded deployment or educational use.
"""
print(memo)

In [ ]:
# Solution P2: Non-technical pitch (example for the literature startup)

pitch = f"""
=== 60-SECOND PITCH ===

We trained a model that read all of Shakespeare's plays and learned the
rhythms, vocabulary, and dramatic style, right down to how letters and words
follow each other in his writing.

Now, give it any opening (a character's name, the start of a speech)
and it continues in Shakespeare's voice, automatically.

Here's what it writes when you give it just "ROMEO:" as a start:
  \"{gen_b_finetuned[:200]}...\"
"""
print(pitch)

## Final Comparison: Path A vs Path B

In [ ]:
# Side-by-side comparison table
import pandas as pd

comparison = pd.DataFrame({
    'Attribute':   [
        'Model architecture',
        'Trainable parameters',
        'Token granularity',
        'Vocab size',
        'Training time (T4 GPU)',
        'Saved model size',
        'Evaluation metric',
        'Training from scratch?',
        'Code complexity',
        'Output coherence',
        'Recommended for',
    ],
    'Path A: Char GRU': [
        'Embedding + GRU + Linear',
        f'{sum(p.numel() for p in model_a.parameters()):,}',
        'Character',
        f'{vocab_size} chars',
        f'~{train_time_a/60:.0f} min',
        f'{model_size_a:.1f} MB',
        f'BPC = {bpc_val_a:.3f}',
        'Yes - all weights random init',
        'Low - PyTorch from scratch',
        'Good at local patterns; occasionally incoherent',
        'Learning, edge deployment, full control',
    ],
    'Path B: GPT-2': [
        '12-layer transformer (GPT-2)',
        '117,000,000+',
        'Subword BPE',
        '50,257 subwords',
        f'~{train_time_b/60:.0f} min',
        f'~{model_size_b_mb:.0f} MB',
        f'Perplexity = {perplexity_b:.1f}',
        'No - pretrained weights shifted',
        'High - HuggingFace Trainer API',
        'Higher - leverages pretrained world knowledge',
        'Production quality, style transfer, demos',
    ],
})

comparison = comparison.set_index('Attribute')
print(comparison.to_string())

In [ ]:
# Side-by-side sample outputs
print("=" * 70)
print("SEED: 'ROMEO:'")
print("=" * 70)
print()
print("PATH A (Char-Level GRU):")
print(gen_a[:300])
print()
print("PATH B (Fine-tuned GPT-2):")
print(gen_b_finetuned[:300])

## Self-Check Quiz: Answers

**Q1: You have 1.1 MB of text. Would you rather train a char-level LSTM from scratch or fine-tune GPT-2?**

Both work on this dataset, but for different reasons:
- Char-level GRU from scratch works because 1.1 MB is ~1M characters, which is enough for a small model to learn char-level patterns. The vocabulary is just 65 tokens, so the problem is tractable.
- Fine-tuning GPT-2 works even better because the model already "knows" English grammar, many Shakespearean words, and general writing structure. You are just *shifting its style*, not teaching it language from scratch.

If you had only 10 KB of text, fine-tuning GPT-2 would win decisively.

**Q2: In Path A, what is the expected initial cross-entropy loss at random init?**

At random initialization, the model assigns approximately equal probability to all `vocab_size` characters. The cross-entropy for a uniform distribution over V classes is `ln(V)`.

For Shakespeare: `ln(65) ~ 4.17 nats`. You should see your epoch-1 training loss start near this value and decrease from there.

**Q3: Your Path A output is `"aaabbbcccddd..."`. Name two likely bugs.**

1. Temperature too low (e.g., `temp=0.01`): the model always picks the single most likely character, which at char-level often means repeating vowels or common letters.
2. Not resetting the hidden state properly between generation steps: the hidden state from a previous training batch is leaking into generation and biasing the output.
3. (Bonus) Loss not decreasing: the model didn't train at all (e.g., forgot `optimizer.step()` or `loss.backward()`), so it generates near-random output that happens to cluster around the most frequent chars.

**Q4: Your Path B output is indistinguishable from base GPT-2. What does this tell you about your fine-tune?**

The fine-tuning had essentially no effect. Likely causes:
- LR too high and the fine-tuning step overwrote the pretrained weights completely (catastrophic forgetting), then the model re-converged to something similar.
- LR too low (near zero) and the weights barely moved.
- Too few steps: the dataset was chunked incorrectly and only a handful of batches were created.
- Trainer evaluated the wrong model: the fine-tuned model wasn't moved to the right device before generation, and the base model (still in memory) was called instead.

## Course Wrap-up

You have completed the full ML & NLP course by Data Trainers LLC.

| Module | Skills |
|--------|--------|
| 1: Pre-NLP | Topic modeling (LDA), NER with spaCy, TF-IDF, logistic regression, boosting |
| 2: Text Similarity | Word2Vec CBOW from scratch, Doc2Vec, sentence-transformers fine-tuning |
| 3: Text Classification | MLP with learned embeddings, BERT fine-tuning with HuggingFace Trainer |
| 4: Text Generation | Char-level GRU, temperature sampling, GPT-2 fine-tuning, top-p generation |

### Where to Go Next

- **foundations-genai**: PyTorch tensor ops, autograd, GPU, DataLoader deep dives.
- **chatbots-with-genai**: LangChain, LangGraph, RAG, conversational agents.
- **genai_and_llms**: LoRA, PEFT, quantization, prompt engineering at scale.